# TV2 – Chuẩn hóa dữ liệu & PCA 2D | Nhóm 14

**Thành viên:** Huy

**Đề tài:** WiFi Localization (D08)

**Môi trường:** Python / Google Colab

**Thực thi:** Runtime → Run all.

Nhãn `Room` chỉ được sử dụng để tô màu biểu đồ,
không tham gia quá trình huấn luyện PCA.


## 2. Chuẩn hóa dữ liệu và giảm chiều bằng PCA

### 2.1. Mục tiêu

Biểu diễn dữ liệu WiFi Localization từ không gian
7 đặc trưng xuống không gian 2 chiều bằng PCA,
phục vụ trực quan hóa sự phân bố của 4 phòng.

### 2.2. Chuẩn hóa dữ liệu

Dữ liệu gồm 2.000 mẫu, 7 đặc trưng tín hiệu WiFi
và một nhãn Room nhận các giá trị 1, 2, 3, 4.

Sử dụng StandardScaler để chuẩn hóa từng đặc trưng
về trung bình bằng 0 và độ lệch chuẩn bằng 1.

Việc chuẩn hóa cần thiết vì các đặc trưng có
thang biến thiên khác nhau. Cụ thể:

- WiFi_1 có độ lệch chuẩn gấp 3,3127 lần WiFi_2.
- WiFi_4 có độ lệch chuẩn gấp 3,3567 lần WiFi_2.
- Phương sai tương ứng lớn gấp khoảng 11 lần.

Nếu không chuẩn hóa, các thuộc tính có phương sai
lớn có thể chi phối hướng thành phần chính của PCA.

### 2.3. Áp dụng PCA

Sử dụng Pipeline gồm:

1. StandardScaler()
2. PCA(n_components=2)

Nhãn Room không được đưa vào quá trình PCA.

Kết quả:

- PC1 giải thích 48,5077% phương sai.
- PC2 giải thích 24,4537% phương sai.
- Tổng phương sai giữ lại đạt 72,9614%.
- Phương sai không được biểu diễn trong 2 thành
  phần chính chiếm khoảng 27,0386%.

### 2.4. Nhận xét trực quan

Biểu đồ PCA 2D cho thấy dữ liệu của bốn phòng
hình thành các vùng phân bố tương đối khác biệt.

Phòng 2 tập trung chủ yếu ở phía PC1 dương,
trong khi Phòng 4 tập trung ở phía PC1 âm.

Phòng 1 có xu hướng nằm ở vùng PC2 âm, còn
Phòng 3 tập trung nhiều hơn quanh vùng trung tâm.

Tuy nhiên, vẫn tồn tại các vùng chồng lấn giữa
một số phòng, bao gồm khu vực ranh giới giữa
Phòng 2 và Phòng 3.

Điều này cho thấy tín hiệu WiFi có khả năng
phản ánh sự khác biệt vị trí, nhưng không thể
phân tách hoàn toàn các phòng chỉ bằng 2 thành
phần chính.

### 2.5. Kết luận

PCA giảm số chiều từ 7 xuống 2 và giữ lại
khoảng 72,96% tổng phương sai của dữ liệu
đã chuẩn hóa.

Kết quả hỗ trợ trực quan hóa cấu trúc dữ liệu,
nhưng không trực tiếp chứng minh độ chính xác
của mô hình phân lớp.

## Phần thực thi – Python

Các cell Python được xây dựng từ
`scripts/pca_visualization.py`.

Notebook thực hiện chuẩn hóa dữ liệu, giảm chiều,
trực quan hóa và xuất kết quả phân tích.


In [ ]:
"""TV2 - Huy: Chuẩn hóa dữ liệu và trực quan hóa PCA 2D."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_project_root():
    current = Path.cwd()
    for folder in (current, *current.parents):
        excel = folder / "data/raw/D08 - wifi_localization.xlsx"
        if excel.is_file():
            return folder
    return current

ROOT = find_project_root()
EXCEL_PATH = ROOT / "data/raw/D08 - wifi_localization.xlsx"
CSV_PATH = ROOT / "data/processed/wifi_localization.csv"
OUTPUT_DIR = ROOT / "figures/pca"

FEATURES = [f"WiFi_{i}" for i in range(1, 8)]
TARGET = "Room"
COLUMNS = FEATURES + [TARGET]
RANDOM_STATE = 42

### A. Nạp và kiểm tra dữ liệu

Đọc file Excel với `header=None` nhằm bảo toàn
đầy đủ 2.000 mẫu dữ liệu.

Nếu file CSV đã tồn tại, chương trình đối chiếu
dữ liệu và thứ tự dòng với Excel gốc.


In [ ]:
def load_data():
    """Đọc đúng sheet, giữ mẫu đầu và không ghi đè CSV khác dữ liệu gốc."""
    if not EXCEL_PATH.is_file():
        from urllib.request import urlopen

        raw_url = (
            "https://raw.githubusercontent.com/Minhhieu3012/"
            "wifi-localization-data-mining/main/data/raw/"
            "D08%20-%20wifi_localization.xlsx"
        )

        try:
            with urlopen(raw_url, timeout=40) as response:
                excel_bytes = response.read()

            EXCEL_PATH.parent.mkdir(
                parents=True, exist_ok=True
            )
            EXCEL_PATH.write_bytes(excel_bytes)

            print("Da tai Excel goc tu GitHub:", EXCEL_PATH)

        except Exception as exc:
            raise FileNotFoundError(
                "Khong tim thay Excel va khong tai duoc GitHub. "
                "Kiem tra Internet hoac thu muc data/raw."
            ) from exc

    raw = pd.read_excel(
        EXCEL_PATH,
        sheet_name="wifi_localization",
        header=None,
    )

    if raw.shape != (2000, 8):
        raise ValueError(f"Excel có kích thước {raw.shape}, cần kiểm tra với TV1.")

    raw.columns = COLUMNS
    raw = raw.apply(pd.to_numeric, errors="raise")

    if not np.isfinite(raw.to_numpy(dtype=float)).all():
        raise ValueError("Dữ liệu có giá trị thiếu hoặc vô hạn.")

    if not raw[TARGET].isin([1, 2, 3, 4]).all():
        raise ValueError("Nhãn phòng phải thuộc 1, 2, 3, 4.")

    raw[TARGET] = raw[TARGET].astype("int64")
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

    if CSV_PATH.exists():
        existing = pd.read_csv(CSV_PATH)

        if list(existing.columns) != COLUMNS:
            raise ValueError("Header CSV khác quy ước. Cần đối chiếu với TV1.")

        existing = existing.apply(pd.to_numeric, errors="raise")

        if existing.shape != raw.shape or not np.array_equal(
            existing.to_numpy(dtype=float),
            raw.to_numpy(dtype=float),
        ):
            raise ValueError(
                "CSV khác Excel về dữ liệu hoặc thứ tự dòng. "
                "Không ghi đè; cần thống nhất với TV1."
            )
    else:
        raw.to_csv(CSV_PATH, index=False, encoding="utf-8")

    # Dùng CSV làm đầu vào phân tích theo yêu cầu đề.
    return pd.read_csv(CSV_PATH)

### B. Hàm xuất biểu đồ

Các biểu đồ được xuất dưới hai định dạng:

- **PNG:** Phục vụ chèn hình ảnh vào báo cáo.
- **PDF:** Giữ chất lượng đồ họa khi phóng to.


In [ ]:
def save_figure(fig, filename):
    """Xuất cả PNG để chèn báo cáo và PDF để giữ chất lượng hình."""
    for extension in ("png", "pdf"):
        fig.savefig(
            OUTPUT_DIR / f"{filename}.{extension}",
            dpi=220,
            bbox_inches="tight",
        )
    plt.close(fig)

### C. Chuẩn hóa và giảm chiều bằng PCA

Áp dụng Pipeline gồm:

1. `StandardScaler()`: Chuẩn hóa 7 đặc trưng WiFi.
2. `PCA(n_components=2)`: Giảm xuống 2 thành phần chính.

Nhãn `Room` không được sử dụng để tính PCA.

Kết quả PCA được lưu thành các bảng thống kê,
tọa độ hai chiều và biểu đồ.


In [ ]:
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})

    df = load_data()
    X = df[FEATURES].astype("float64")
    y = df[TARGET].astype("int64")

    # Nhãn Room không tham gia bước chuẩn hóa hoặc PCA.
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(
            n_components=2,
            svd_solver="full",
            random_state=RANDOM_STATE,
        )),
    ])

    coordinates = pipeline.fit_transform(X)
    scaler = pipeline.named_steps["scaler"]
    pca = pipeline.named_steps["pca"]
    scaled = scaler.transform(X)
    ratios = pca.explained_variance_ratio_

    # Kiểm tra kết quả chuẩn hóa và kích thước đầu ra.
    assert coordinates.shape == (len(df), 2)
    assert np.isfinite(coordinates).all()
    assert np.allclose(scaled.mean(axis=0), 0, atol=1e-10)
    assert np.allclose(scaled.std(axis=0), 1, atol=1e-10)

    statistics = pd.DataFrame({
        "mean_raw": X.mean(),
        "std_raw": X.std(ddof=0),
        "variance_raw": X.var(ddof=0),
        "mean_scaled": scaled.mean(axis=0),
        "std_scaled": scaled.std(axis=0),
    })
    statistics.to_csv(
        OUTPUT_DIR / "pca_scaling_statistics.csv",
        index_label="feature",
    )

    variance = pd.DataFrame({
        "component": ["PC1", "PC2"],
        "explained_variance_ratio": ratios,
        "cumulative_ratio": np.cumsum(ratios),
    })
    variance.to_csv(OUTPUT_DIR / "pca_variance.csv", index=False)

    # sample_id đánh số theo thứ tự mẫu trong CSV, bắt đầu từ 1.
    points = pd.DataFrame({
        "sample_id": np.arange(1, len(df) + 1),
        "PC1": coordinates[:, 0],
        "PC2": coordinates[:, 1],
        "Room": y.to_numpy(),
    })
    points.to_csv(OUTPUT_DIR / "pca_coordinates.csv", index=False)

    # Biểu đồ tỷ lệ phương sai giữ lại.
    fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
    bars = ax.bar(
        ["PC1", "PC2"],
        ratios * 100,
        color=["#0072B2", "#009E73"],
    )
    ax.bar_label(bars, labels=[f"{r:.2%}" for r in ratios], padding=5)
    ax.set_ylim(0, max(ratios) * 100 + 12)
    ax.set_ylabel("Phương sai giải thích (%)")
    ax.set_title(f"PCA — Tổng phương sai giữ lại: {ratios.sum():.2%}")
    save_figure(fig, "pca_explained_variance")

    # Scatter plot: mỗi phòng có màu và marker riêng.
    colors = ["#0072B2", "#E69F00", "#009E73", "#CC79A7"]
    markers = ["o", "s", "^", "D"]
    fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)

    for room, color, marker in zip(range(1, 5), colors, markers):
        mask = y.to_numpy() == room
        ax.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            color=color,
            marker=marker,
            s=23,
            alpha=0.55,
            linewidths=0,
            label=f"Phòng {room} (n={int(mask.sum())})",
        )

    ax.set_xlabel(f"PC1 ({ratios[0]:.2%} phương sai)")
    ax.set_ylabel(f"PC2 ({ratios[1]:.2%} phương sai)")
    ax.set_title("WiFi Localization — StandardScaler + PCA 2D")
    ax.grid(alpha=0.15)
    ax.set_axisbelow(True)
    ax.legend(title="Nhãn thực tế")
    save_figure(fig, "pca_rooms_2d")

    print(f"Dữ liệu: {len(df)} mẫu, {X.shape[1]} đặc trưng")
    print(f"PC1: {ratios[0]:.4%}")
    print(f"PC2: {ratios[1]:.4%}")
    print(f"Tổng phương sai giữ lại: {ratios.sum():.4%}")
    print("\nThống kê chuẩn hóa:")
    print(statistics.round(6).to_string())

    for feature in ("WiFi_1", "WiFi_4"):
        ratio = statistics.loc[feature, "std_raw"] / statistics.loc["WiFi_2", "std_raw"]
        print(
            f"\n{feature}/WiFi_2: "
            f"tỷ số độ lệch chuẩn = {ratio:.4f}; "
            f"tỷ số phương sai = {ratio ** 2:.4f}"
        )

    print(f"\nHoàn thành. Kết quả lưu tại: {OUTPUT_DIR}")

### D. Thực thi và trực quan hóa kết quả

Chạy toàn bộ quy trình phân tích.

Kết quả bao gồm:

- Bảng thống kê dữ liệu trước và sau chuẩn hóa.
- Tỷ lệ phương sai giải thích của PC1 và PC2.
- Biểu đồ tỷ lệ phương sai giữ lại.
- Biểu đồ PCA 2D của bốn phòng.
- Các file CSV, PNG và PDF để bàn giao.


In [ ]:
main()

from IPython.display import Image, display

print("\n=== PCA EXPLAINED VARIANCE ===")
display(Image(
    filename=str(OUTPUT_DIR / "pca_explained_variance.png")
))

print("\n=== PCA 2D THEO PHONG ===")
display(Image(
    filename=str(OUTPUT_DIR / "pca_rooms_2d.png")
))

expected = [
    "pca_variance.csv",
    "pca_scaling_statistics.csv",
    "pca_coordinates.csv",
    "pca_explained_variance.png",
    "pca_explained_variance.pdf",
    "pca_rooms_2d.png",
    "pca_rooms_2d.pdf"
]

missing = [
    name for name in expected
    if not (OUTPUT_DIR / name).exists()
]

assert not missing, f"Thieu file ket qua: {missing}"

print("\nDA XUAT DU 7 FILE:")
for name in expected:
    print(" -", OUTPUT_DIR / name)

## Kết luận và lưu ý khi tích hợp bài nhóm

PCA giảm không gian đặc trưng từ 7 chiều xuống
2 chiều và giữ lại khoảng **72,96% tổng phương sai**
của dữ liệu đã chuẩn hóa.

Biểu đồ cho thấy bốn phòng có sự khác biệt về
phân bố tín hiệu WiFi, nhưng vẫn tồn tại
một số vùng chồng lấn.

**Lưu ý:**

- `Room` không tham gia quá trình fit PCA.
- Tỷ lệ phương sai giữ lại không phải Accuracy
  hoặc F1-score của mô hình phân lớp.
- PCA 2D phục vụ trực quan hóa dữ liệu.
- Khi áp dụng biến đổi dữ liệu trong đánh giá
  mô hình bằng cross-validation, các bước
  cần được fit bên trong từng fold để tránh
  data leakage.
- File CSV dùng chung cần thống nhất với TV1.
